## Longest track task
### Loading parameters

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import track_builder as tb
import os

data = r"D:\Stockage\ASTD"
parquet_path = "../data/"

year = 2019
months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

# Periods for load_periods
periods = {
    2019: months,
    2020: months
}

### Loading functions
Load raw csv files and save dataframe as parquet file - (it's faster to load parquet files)\
If parquet already exists, load the parquet file as a dataframe

In [2]:
def load_data(parquet_file, source, year, **kwargs):
    parquet_file = parquet_path + parquet_file
    if os.path.exists(parquet_file):
        data =  pd.read_parquet(parquet_file)
        print(f"Loaded data {parquet_file}, parameters ignored")
    else:
        data = tb.load_astd_monthly(base_path=source, year=year, **kwargs)
        data.to_parquet(parquet_file)
        print(f"Loaded data {parquet_file} from {source}")

    return data

def load_tracks(parquet_file, data, **kwargs):
    parquet_file = parquet_path + parquet_file
    if os.path.exists(parquet_file):
        tracks =  pd.read_parquet(parquet_file)
        print(f"Loaded tracks {parquet_file}, parameters ignored")
    else:
        tracks = tb.build_ship_tracks(data, **kwargs)
        tracks.to_parquet(parquet_file)
        print(f"Loaded tracks {parquet_file} from {data}")

    return tracks

def load_periods(parquet_file, source, periods, **kwargs):
    parquet_file = parquet_path + parquet_file
    if os.path.exists(parquet_file):
        period =  pd.read_parquet(parquet_file)
        print(f"Loaded period {parquet_file}, parameters ignored")
    else:
        period = tb.load_astd_periods(base_path=source, periods=periods, **kwargs)
        period.to_parquet(parquet_file)
        print(f"Loaded period {parquet_file} from {source}")

    return period

### Import all segments of selected year

Optional: \
Import segments first and last day to build tracks\
optional because first and last day are automatically recovered in algo, this could potentially make the process faster

In [3]:
all_data = load_data(parquet_file = f'all_segments{year}.parquet', source = data, year = year, months = months, remove_nan_rows="default", usecols="default", progress=True)
all_data.sample(5)

# Optional
# spe_track = load_data(parquet_file = f'spefirstlast{year}.parquet', source = data, year = year, months = months, remove_nan_rows="default", usecols="default", sampling=[0, -1], progress=True)
# spe_track.sample(5)

Loaded data ../data/all_segments2019.parquet, parameters ignored


,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude
27895412,3400,2019-09-05 00:29:16+00:00,Russia,FS Ice Class 1B,Fishing vessels,1000 - 4999 GT,711.190735,370,12.547525,76.972908
33351352,3317,2019-10-19 19:32:56+00:00,Norway,FS Ice Class 1C,Offshore supply ships,1000 - 4999 GT,2.706846,5,15.420205,68.699387
24447290,4389,2019-08-09 23:37:25+00:00,Iceland,FS Ice Class 1C,Fishing vessels,1000 - 4999 GT,5.574742,99,-13.738880,65.137627
18968277,2049,2019-06-30 15:18:10+00:00,Russia,FS Ice Class II,Fishing vessels,< 1000 GT,4581.068359,976,47.980419,76.585922
19446563,3122,2019-07-04 08:19:01+00:00,Finland,FS Ice Class 1A Super,Passenger ships,25000 - 49999,3775.314209,389,21.087446,60.084686


### Build ship tracks
Build tracks between each months' segments with last day of previous month and first day of next month

In [4]:
tracks = load_tracks(parquet_file = f'tracks{year}.parquet', data = all_data)
tracks.sample(5)

Loaded tracks ../data/tracks2019.parquet, parameters ignored


,month,segment_id,track_id
8232,2019-08,14443,4318
3037,2019-06,3404,2109
456,2019-01,3013,457
6826,2019-07,4639,3675
3330,2019-09,3476,2193


### Build tracks position

'Ghost' point removal:\
remove_unrealistic_points

Horizontal date lines jumps:\
mask_dateline_jumps

Region selection:\
Updating region selection to handle multiple regions

In [93]:
# Create a month field to avoid build_light_multi_track_data to hang
all_data['month'] = all_data['date_time_utc'].dt.to_period('M').astype(str)

#Build tracks positions - with region selection
build_tracks = tb.build_light_multi_track_data(tracks, track_sampling=100, positions_df=all_data, region=["canada", "iceland", "russia", "norway", "usa"], minimal_region=['russia', 'norway'],preprocess_positions=True)
display(build_tracks)

C:\Users\virtu\AppData\Local\Temp\ipykernel_30556\767548245.py:2: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.



computing typical speeds...


D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:382: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

D:\Projets\clone\TrackBuilder\track_builder\core\track_helpers.py:170: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



Cleaning completed: 1420 'ghost' or aberrant points removed.


D:\Projets\clone\TrackBuilder\track_builder\io\astd_loader.py:185: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,track_id,region
0,3273,2019-07-10 18:03:22+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,2015.969727,362,-179.734711,65.031532,2019-07,4293,russia
1,3273,2019-07-10 21:41:50+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,11982.660156,1889,-179.085205,65.623230,2019-07,4293,russia
2,3273,2019-07-11 00:30:45+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,9079.882812,1769,-179.134171,66.145493,2019-07,4293,russia
3,3273,2019-07-11 08:28:39+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,12.820436,539,-179.112686,66.314690,2019-07,4293,russia
4,3273,2019-07-11 12:49:35+00:00,Russia,FS Ice Class 1A,General cargo ships,5000 - 9999 GT,6.039132,1080,-179.112732,66.314606,2019-07,4293,russia
...,...,...,...,...,...,...,...,...,...,...,...,...,...
991,2162,2019-12-26 02:04:05+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,4.350543,364,33.032345,68.958641,2019-12,5274,norway
992,2162,2019-12-27 21:17:31+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.952028,177,33.032425,68.958694,2019-12,5274,norway
993,2162,2019-12-28 21:25:24+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.349052,363,33.032413,68.958687,2019-12,5274,norway
994,2162,2019-12-30 05:13:25+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,2.941393,7,33.032318,68.958641,2019-12,5274,norway


### Longest track function

In [94]:
def get_longest_tracks(tracks : pd.DataFrame, n_tracks : int =5):
    # Verify column name exists
    if not {'track_id', 'dist_nextpoint'}.issubset(tracks.columns):
        print("No tracks found")
        return None

    # Get the total distance for each tracks
    total_dist_ship = (
        tracks.groupby('track_id', as_index=False)['dist_nextpoint']
        .sum()
        .rename(columns={'dist_nextpoint': 'total_distance'})
        .sort_values('total_distance', ascending=False)
    )

    df_longest_tracks = total_dist_ship.head(n_tracks)
    longest_track_ids = df_longest_tracks['track_id'].to_list()

    print("Longest track segments")
    display(tracks[tracks['track_id'] == longest_track_ids[0]])

    print('Longest tracks')
    display(df_longest_tracks)

    return longest_track_ids

In [95]:
longest_track_id = None
print("Minimal corresponding tracks", build_tracks['track_id'].nunique())
#
if build_tracks['track_id'].nunique() > 0:
    # Compute longest_tracks
    longest_track_ids = get_longest_tracks(build_tracks)


Minimal corresponding tracks 3
Longest track segments


,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,track_id,region
537,3271,2019-09-01 13:25:15+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,4830.845703,782,50.106907,70.026428,2019-09,5274,russia
538,3271,2019-09-01 18:19:22+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,2643.692383,430,52.643909,70.086609,2019-09,5274,russia
539,3271,2019-09-01 20:32:41+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,19762.056641,3123,53.976292,70.118164,2019-09,5274,russia
540,3271,2019-09-02 02:13:21+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1351.466553,212,57.432674,70.210846,2019-09,5274,russia
541,3271,2019-09-02 07:15:18+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,29079.544922,4651,58.825012,70.723679,2019-09,5274,russia
...,...,...,...,...,...,...,...,...,...,...,...,...,...
991,2162,2019-12-26 02:04:05+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,4.350543,364,33.032345,68.958641,2019-12,5274,norway
992,2162,2019-12-27 21:17:31+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.952028,177,33.032425,68.958694,2019-12,5274,norway
993,2162,2019-12-28 21:25:24+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,1.349052,363,33.032413,68.958687,2019-12,5274,norway
994,2162,2019-12-30 05:13:25+00:00,Russia,FS Ice Class 1A Super,Other activities,1000 - 4999 GT,2.941393,7,33.032318,68.958641,2019-12,5274,norway


Longest tracks


,track_id,total_distance
2,5274,1.463558e+06
0,4293,1.228853e+06
1,5026,5.630048e+05


### Visualize longest tracks

In [97]:
# Show special track region coverage
display(build_tracks[build_tracks['track_id'] == 4293]['region'].unique())
# display(build_tracks[build_tracks['track_id'] == longest_track_id]['sel_region'].unique())

# Clean horizontal lines
# build_tracks = tb.core.track_helpers.mask_dateline_jumps(build_tracks)

fig = tb.plot_ship_tracks(
    build_tracks,
    track_ids=longest_track_ids,
    color_by="track_id",
    color_mode="categorical",
    show_start_end=True,
    map_style="open-street-map",
    title="Longest track"
    )
fig.update_layout(showlegend=False)
fig.show()

array(['russia', 'usa', 'norway'], dtype=object)

In [100]:
display(build_tracks[build_tracks['track_id'] == 4293]['region'].unique())
usa_track = build_tracks[(build_tracks['region'] == "usa") & (build_tracks['track_id'] == 4293)]

fig = tb.plot_ship_tracks(
    usa_track,
    track_ids=longest_track_ids + [4293],
    color_by="track_id",
    color_mode="categorical",
    show_start_end=True,
    map_style="open-street-map",
    title="Longest track"
    )
fig.update_layout(showlegend=False)
fig.show()

array(['russia', 'usa', 'norway'], dtype=object)